<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-2/07-segmentacion-de-clientes/solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Solución 2.7 · Segmentación de clientes

**Serie:** Del Excel al Machine Learning · **Temporada 2** · 🎬 Video próximamente

Estas son las respuestas del ejercicio. Si la tuya es distinta pero da el mismo resultado, ¡también está bien! En programación suele haber más de una forma de llegar a la respuesta.

### Prepara los datos

Ejecuta estas celdas. Dejan listas `ventas`, `clientes` y `resumen`. `resumen` debe tener `300` filas.

Algunos ejercicios usan variables de los anteriores, así que **ejecútalos en orden**.

In [1]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
def resumen_de_clientes(ventas):
    """Una fila por cliente: días sin comprar, y pedidos y compra por mes activo."""
    fecha = pd.to_datetime(ventas["fecha"])
    tabla = (
        ventas.assign(fecha=fecha)
        .groupby("id_cliente")
        .agg(
            primera=("fecha", "min"),
            ultima=("fecha", "max"),
            pedidos=("fecha", "nunique"),
            total=("total", "sum"),
        )
    )
    meses = ((tabla["ultima"] - tabla["primera"]).dt.days / 30.44).clip(lower=1)
    return pd.DataFrame(
        {
            "dias_sin_comprar": (pd.Timestamp("2025-12-31") - tabla["ultima"]).dt.days,
            "pedidos_mes": (tabla["pedidos"] / meses).round(1),
            "compra_mes": (tabla["total"] / meses).round(1),
        }
    )

In [3]:
ventas = pd.read_csv(DATOS + "ventas.csv")
clientes = pd.read_csv(DATOS + "clientes.csv").set_index("id_cliente")

resumen = resumen_de_clientes(ventas)

print(len(resumen))

300


### 1. Tres grupos

Escala `resumen` y arma 3 grupos con `n_init=10` y `random_state=0`. Guárdalos en `grupos`. Muestra el promedio de cada columna por grupo y cuántos clientes tiene cada uno.

In [4]:
escalados = StandardScaler().fit_transform(resumen)
grupos = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(escalados)

resumen.groupby(grupos).mean().round(1).assign(clientes=resumen.groupby(grupos).size())

,dias_sin_comprar,pedidos_mes,compra_mes,clientes
0,10.3,11.3,4694.6,57
1,5.2,5.2,1409.9,217
2,397.0,7.2,2513.5,26


### 2. Ponles nombre

Responde en un comentario. ¿Qué nombre le pondrías a cada uno de tus 3 grupos? ¿Qué harías distinto con cada uno?

In [5]:
# Por ejemplo: "perdidos", a quienes se llama para saber qué pasó; "chicos", con un pedido mínimo
# más simple; y "grandes", con una atención prioritaria. Otros nombres también valen.

### 3. Comparar con el tipo

Cruza `grupos` con el `tipo` de cliente, usando `pd.crosstab()` como en la clase. ¿Los grupos repiten el tipo?

In [6]:
pd.crosstab(grupos, clientes["tipo"].reindex(resumen.index))

# Se parecen, pero no repiten el tipo.

tipo,Bodega,Minimarket,Restaurante
row_0,,,
0,13,36,8
1,155,29,33
2,12,10,4


### 4. La regla del 1.9

Cuenta cuántos clientes tienen `dias_sin_comprar` mayor que 91, es decir, que no compraron de octubre a diciembre. Deben ser `30`.

In [7]:
perdidos = resumen[resumen["dias_sin_comprar"] > 91]

len(perdidos)

30

### 5. El gran cliente perdido

Entre esos clientes, ¿cuál tiene la mayor `compra_mes`? Debe ser `C0105`. Usa `.sort_values()`.

In [8]:
perdidos.sort_values("compra_mes", ascending=False).head(3)

,dias_sin_comprar,pedidos_mes,compra_mes
id_cliente,,,
C0105,508,16.8,8658.6
C0254,543,11.5,7023.6
C0113,511,14.7,6796.3


### 6. ¿Qué agregarías?

Responde en un comentario. Si tuvieras que agregar una cuarta columna a `resumen`, ¿cuál sería, y qué decisión de negocio ayudaría a tomar?

In [9]:
# Por ejemplo, la parte de la compra al crédito, para decidir a qué grupo ofrecer más plazo.

## 🎉 ¡Bien hecho!

Nos vemos en el **episodio 2.8: Ética I: auditar una decisión**.

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0